### 1. 그래프의 상태 업데이트
HumanMessage : 사용자(사람)의 메시지 

AIMessage : AI(LLM)의 메시지

AnyMessage : HumanMessage, AIMessage를 포함하는 메시지

ToolMessage: 검색한 결과를 나타내는 메세지



```
딕셔너리 타입
state = {
    "messages" : [],
    "step": 1
}

TypedDict{
    "messages": list,
    "step": int
}
```

### 2. 왜 랭그래프에서 TypedDict을 사용하는 이유
- 랭그래프에서는 State Graph라는 개념이 존재
```
# 노드+노드+노드 -> 이런식으로 연결이 가능
```
- state가 계속 전달된다.

```
# state에는 messages라는 key가 있고, 값은 AnyMessage리스트다.

class state(TypeDict):
    messages: list[AnyMessage] -> 이런식으로 타입을 명확히 전달
```

In [ ]:
from langchain_core.messages import AnyMessage # 랭체인이 당연히 포함
from typing_extensions import TypedDict

class State(TypedDict):
    messages: list[AnyMessage]
    extra_field: int

In [ ]:
from langchain_core.messages import AIMessage

def node(state: State): # 노드 함수에 State값을 받아서 
    messages = state["messages"] # 메세지 속성을 가져와 안에 넣었다가
    new_message = AIMessage("안녕하세요! 무엇을 도와드릴까요?") # AI메세지 안에 새로운 메세지를 만들어서

    # return {"messages": new_message, "extra_field": 10} 
    return {"messages": messages + [new_message], "extra_field": 10} 
# 메세지 속성에 기존 메세지와 새로운 메세지를 더해서 반환한다. 
# extra_field는 10으로 설정한다. 그냥 큰 의미는 없다.

### 상태기반 그래프
StateGraph(State) = state based graph 

이걸로 노드가 연결되는 식으로 chain이 형성된다.

In [ ]:
from langgraph.graph import StateGraph

graph_builder = StateGraph(State) # 랭그래프에서 상태 기반 그래프
graph_builder.add_node("node", node)
# set_entry_point : 그래프의 시작 노드를 지정하는 엣지 (START -> "node")
graph_builder.set_entry_point("node")
graph = graph_builder.compile()

graph